# Benchmark: Scalability vs Grid Size

Measures how correction runtime, L2 error, and iteration count scale with
increasing grid resolution.  A fixed random DVF pattern is generated at a
small base size and upscaled to each target resolution, keeping the folding
pattern consistent across sizes.

Sizes tested: **5x5, 8x8, 10x10, 12x12, 15x15, 18x18, 20x20**

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from dvfopt import (
    iterative_with_jacobians2,
    iterative_parallel,
    jacobian_det2D,
    generate_random_dvf,
    scale_dvf,
)
from benchmark_utils import (
    run_correction,
    plot_jac_heatmaps,
    plot_correction_magnitude,
    plot_jdet_histograms,
)

## Configuration

In [ ]:
GRID_SIZES = [5, 8, 10, 12, 15, 18, 20]

# Base DVF: small field upscaled to each target size
BASE_SHAPE = (3, 1, 5, 5)
MAX_MAGNITUDE = 5.0
SEED = 42

# Use parallel solver for larger grids
USE_PARALLEL = True

## Generate test fields at each resolution

In [ ]:
base_dvf = generate_random_dvf(BASE_SHAPE, MAX_MAGNITUDE, SEED)

test_fields = {}
for size in GRID_SIZES:
    dvf = scale_dvf(base_dvf, (size, size))
    phi = np.stack([dvf[-2, 0], dvf[-1, 0]])
    jac = jacobian_det2D(phi)
    n_neg = int((jac <= 0).sum())
    test_fields[size] = dvf
    print(f"  {size:>4d}x{size:<4d}  neg-Jdet: {n_neg:>5d}  "
          f"min Jdet: {jac.min():+.4f}  total pixels: {size*size}")

## Run corrections

In [ ]:
solver = iterative_parallel if USE_PARALLEL else iterative_with_jacobians2
solver_name = "parallel" if USE_PARALLEL else "serial"
print(f"Solver: {solver_name}\n")

results = {}

print(f"{'Size':>10s}  {'Time (s)':>10s}  {'Neg init':>10s}  {'Neg final':>10s}  "
      f"{'Min Jdet':>10s}  {'L2 Error':>10s}")
print("-" * 70)

for size in GRID_SIZES:
    dvf = test_fields[size]
    results[size] = run_correction(dvf, solver)
    r = results[size]
    print(f"{size:>4d}x{size:<4d}  {r['time']:>10.2f}  {r['n_neg_init']:>10d}  "
          f"{r['n_neg_final']:>10d}  {r['min_jdet']:>+10.6f}  {r['l2_err']:>10.4f}")

---
## Jacobian heatmaps — before vs after correction

Each column shows one grid size.  Top row = initial Jacobian, bottom row = corrected.
Red regions have negative Jacobian determinant (folding).

In [ ]:
plot_jac_heatmaps(
    [[results[s]["jac_init"][0] for s in GRID_SIZES],
     [results[s]["jac_final"][0] for s in GRID_SIZES]],
    [f"{s}x{s}" for s in GRID_SIZES],
    title="Jacobian Determinant — Before vs After Correction",
)
plt.show()

## Displacement change — where did the solver move pixels?

Per-pixel magnitude of `|phi_corrected - phi_init|` (how much each pixel was displaced by the correction).

In [ ]:
plot_correction_magnitude(
    [(results[s]["phi"], results[s]["phi_init"]) for s in GRID_SIZES],
    [f"{s}x{s}" for s in GRID_SIZES],
    title="Per-pixel displacement change (correction magnitude)",
)
plt.show()

## Jacobian determinant distributions — before vs after

Histograms show how the Jdet distribution shifts from having a negative tail
(folding) to being entirely positive after correction.

In [ ]:
plot_jdet_histograms(
    [[("Before", results[s]["jac_init"][0]),
      ("After", results[s]["jac_final"][0])] for s in GRID_SIZES],
    [f"{s}x{s}" for s in GRID_SIZES],
    title="Jacobian Determinant Distribution",
)
plt.show()

---
## Scaling plots

In [ ]:
sizes = list(results.keys())
times = [results[s]["time"] for s in sizes]
l2s = [results[s]["l2_err"] for s in sizes]
negs = [results[s]["n_neg_init"] for s in sizes]
pixels = [s * s for s in sizes]
pct_neg = [100 * results[s]["n_neg_init"] / (s * s) for s in sizes]
min_jdet_init = [results[s]["min_jdet_init"] for s in sizes]
min_jdet_final = [results[s]["min_jdet"] for s in sizes]

fig, axes = plt.subplots(3, 2, figsize=(12, 13))

# Runtime vs grid size
ax = axes[0, 0]
ax.plot(sizes, times, "o-", color="tab:blue")
ax.set_xlabel("Grid size (NxN)")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime vs Grid Size")
ax.grid(True, alpha=0.3)

# Runtime vs total pixels (log-log)
ax = axes[0, 1]
ax.loglog(pixels, times, "o-", color="tab:blue")
log_p = np.log(pixels)
log_t = np.log(times)
alpha, log_c = np.polyfit(log_p, log_t, 1)
fit_t = np.exp(log_c) * np.array(pixels) ** alpha
ax.loglog(pixels, fit_t, "--", color="tab:red",
          label=f"fit: O(n^{alpha:.2f})")
ax.set_xlabel("Total pixels")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime Scaling (log-log)")
ax.legend()
ax.grid(True, alpha=0.3)

# L2 error vs grid size
ax = axes[1, 0]
ax.plot(sizes, l2s, "s-", color="tab:orange")
ax.set_xlabel("Grid size (NxN)")
ax.set_ylabel("L2 Error")
ax.set_title("L2 Deviation vs Grid Size")
ax.grid(True, alpha=0.3)

# Initial neg-Jdet count vs grid size
ax = axes[1, 1]
ax.plot(sizes, negs, "^-", color="tab:green", label="Count")
ax.set_xlabel("Grid size (NxN)")
ax.set_ylabel("Neg Jdet pixels (initial)")
ax.set_title("Folding Pixels vs Grid Size")
ax.grid(True, alpha=0.3)
ax2 = ax.twinx()
ax2.plot(sizes, pct_neg, "x--", color="tab:gray", alpha=0.7, label="% of grid")
ax2.set_ylabel("% of total pixels", color="tab:gray")
ax2.tick_params(axis="y", labelcolor="tab:gray")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

# Min Jdet before vs after
ax = axes[2, 0]
x = np.arange(len(sizes))
w = 0.35
ax.bar(x - w / 2, min_jdet_init, w, label="Before", color="tab:red", alpha=0.7)
ax.bar(x + w / 2, min_jdet_final, w, label="After", color="tab:blue", alpha=0.7)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f"{s}x{s}" for s in sizes], rotation=30)
ax.set_ylabel("Min Jacobian determinant")
ax.set_title("Worst-case Jdet: Before vs After")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Time per negative pixel vs grid size
ax = axes[2, 1]
time_per_neg = [results[s]["time"] / max(results[s]["n_neg_init"], 1) for s in sizes]
ax.bar([f"{s}x{s}" for s in sizes], time_per_neg, color="tab:purple")
ax.set_ylabel("Time per neg-Jdet pixel (s)")
ax.set_title("Correction Cost per Folding Pixel")
plt.sca(ax)
plt.xticks(rotation=30)
ax.grid(True, alpha=0.3, axis="y")

plt.suptitle(f"Scalability — {solver_name} solver", fontsize=14)
plt.tight_layout()
plt.show()

## Summary

Key takeaways:
- **Runtime scaling** — the log-log slope gives the empirical complexity exponent
- **L2 error** — larger grids may allow more diffuse corrections (lower per-pixel impact)
- **Cost per pixel** — how much additional work each folding pixel requires at different resolutions